In [1]:


%pip install -q "datasets<4.0.0" transformers accelerate pillow tqdm numpy torch torchvision



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, math, time, random
from dataclasses import dataclass

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW       # use PyTorch AdamW, not transformers [web:34][web:36]
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    get_cosine_schedule_with_warmup,   # still valid in transformers optimization APIs [web:41][web:46]
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

@dataclass
class CFG:
    model_id: str = "Salesforce/blip-image-captioning-base"
    dataset_id: str = "whyen-wang/coco_captions"   # COCO captions dataset: image + list of 5 captions [web:7]

    train_samples: int = 1000     # start small; increase to 10k–50k later
    val_samples: int = 200
    seed: int = 42

    image_size: int = 224
    max_target_len: int = 32

    batch_size: int = 4
    grad_accum: int = 8
    epochs: int = 1

    lr: float = 1e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    max_grad_norm: float = 1.0

    num_workers: int = 0          # safer on macOS
    log_every: int = 10
    save_every_steps: int = 100

    out_dir: str = "./blip_coco_ft_mps"

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
print("✅ Config loaded")


/Users/makumar/Documents/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Config loaded


In [2]:
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_all(cfg.seed)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"✅ Device: {device}")


✅ Device: mps


In [3]:
import aiohttp
import datasets

# Use storage_options to increase the timeout from 5 minutes (300s) to 1 hour (3600s)
ds = load_dataset(
    cfg.dataset_id, 
    trust_remote_code=True,
    storage_options={'client_kwargs': {'timeout': aiohttp.ClientTimeout(total=3600)}}
)

print(ds)
print("Example keys:", ds["train"][0].keys())
print("Captions per image:", len(ds["train"][0]["captions"]))

train_split = "train"
val_split = "validation" if "validation" in ds else ("val" if "val" in ds else "train")

train_ds = ds[train_split].shuffle(seed=cfg.seed).select(
    range(min(cfg.train_samples, len(ds[train_split])))
)
val_ds = ds[val_split].shuffle(seed=cfg.seed + 1).select(
    range(min(cfg.val_samples, len(ds[val_split])))
)

print(f"✅ Train: {len(train_ds)}, Val: {len(val_ds)}")


Generating train split: 118287 examples [00:02, 54322.81 examples/s]
Generating validation split: 5000 examples [00:00, 55846.76 examples/s]


DatasetDict({
    train: Dataset({
        features: ['image', 'captions'],
        num_rows: 118287
    })
    validation: Dataset({
        features: ['image', 'captions'],
        num_rows: 5000
    })
})
Example keys: dict_keys(['image', 'captions'])
Captions per image: 5
✅ Train: 1000, Val: 200


In [4]:
processor = BlipProcessor.from_pretrained(cfg.model_id)
model = BlipForConditionalGeneration.from_pretrained(cfg.model_id)

# Force 224px images (lighter for Mac)
try:
    processor.image_processor.size = {"height": cfg.image_size, "width": cfg.image_size}
except Exception as e:
    print(f"⚠️ Could not set image size: {e}")

# Memory helpers
try:
    model.gradient_checkpointing_enable()
    print("✅ Gradient checkpointing enabled")
except Exception as e:
    print(f"⚠️ Gradient checkpointing failed: {e}")

model.config.use_cache = False   # must be False when using gradient checkpointing
model.to(device)

print(f"✅ Model loaded: {cfg.model_id}")


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 473/473 [00:00<00:00, 1923.98it/s, Materializing param=vision_model.post_layernorm.weight]                                       
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT 

✅ Gradient checkpointing enabled
✅ Model loaded: Salesforce/blip-image-captioning-base


In [9]:
def collate_fn(examples):
    images = [ex["image"].convert("RGB") for ex in examples]
    # pick one random caption per image
    captions = [random.choice(ex["captions"]) for ex in examples]

    encoding = processor(
        images=images,
        text=captions,
        padding="max_length",
        truncation=True,
        max_length=cfg.max_target_len,
        return_tensors="pt",
    )

    # BLIP needs `labels` = `input_ids` for captioning loss
    encoding["labels"] = encoding["input_ids"].clone()

    return encoding


train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)

In [10]:
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

total_update_steps = math.ceil(len(train_loader) / cfg.grad_accum) * cfg.epochs
warmup_steps = int(total_update_steps * cfg.warmup_ratio)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)

print(f"✅ Update steps: {total_update_steps}, Warmup: {warmup_steps}")


✅ Update steps: 32, Warmup: 0


In [11]:
def save_ckpt(step, epoch):
    """
    Save model weights, processor, and training state to cfg.out_dir.
    Directory: out_dir/ckpt_step{step}_epoch{epoch}
    """
    path = os.path.join(cfg.out_dir, f"ckpt_step{step}_epoch{epoch}")
    os.makedirs(path, exist_ok=True)

    # Save model weights + config in HF format
    model.save_pretrained(path)
    processor.save_pretrained(path)

    # Save optimizer/scheduler state, step, epoch
    torch.save(
        {
            "step": step,
            "epoch": epoch,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "cfg": cfg.__dict__,
        },
        os.path.join(path, "train_state.pt"),
    )

    print(f"✅ Checkpoint saved: {path}")


def load_ckpt(path):
    """
    Load model + optimizer/scheduler from a checkpoint directory.
    """
    # Load model weights
    loaded_model = BlipForConditionalGeneration.from_pretrained(path)
    model.load_state_dict(loaded_model.state_dict())

    # Load training state
    state = torch.load(os.path.join(path, "train_state.pt"), map_location="cpu")
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])

    print(f"✅ Resumed from step {state['step']}, epoch {state['epoch']}")
    return state["step"], state["epoch"]


print("✅ Checkpoint helpers ready")


✅ Checkpoint helpers ready


In [12]:
model.train()

global_step = 0
t0 = time.time()

for epoch in range(1, cfg.epochs + 1):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.epochs}")
    running_loss = 0.0

    optimizer.zero_grad(set_to_none=True)

    for i, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        out = model(**batch)          # model returns loss when labels are passed [web:17]
        loss = out.loss / cfg.grad_accum
        loss.backward()

        running_loss += loss.item()

        if i % cfg.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            if global_step % cfg.log_every == 0:
                avg_loss = running_loss / cfg.log_every
                running_loss = 0.0
                pbar.set_postfix({
                    "loss": f"{avg_loss:.4f}",
                    "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                })

            if global_step % cfg.save_every_steps == 0:
                save_ckpt(global_step, epoch)

    # Save checkpoint at end of epoch
    save_ckpt(global_step, epoch)

elapsed = (time.time() - t0) / 60.0
print(f"✅ Training complete in {elapsed:.2f} minutes")


Epoch 1/1:   0%|          | 0/250 [00:00<?, ?it/s]/Users/makumar/Documents/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.02it/s]


✅ Checkpoint saved: ./blip_coco_ft_mps/ckpt_step31_epoch1
✅ Training complete in 1.15 minutes


In [13]:
model.eval()

@torch.no_grad()
def generate_caption(pil_image, max_new_tokens=30, num_beams=3):
    inputs = processor(images=pil_image.convert("RGB"), return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
    )
    return processor.decode(ids[0], skip_special_tokens=True)

print("Sample predictions:\n")
for idx in [0, 1, 2]:
    ex = val_ds[idx]
    gt = ex["captions"][0]
    pred = generate_caption(ex["image"])
    print(f"GT:   {gt}")
    print(f"Pred: {pred}")
    print("-" * 80)

model.train()
print("✅ Inference test complete")


Sample predictions:

GT:   A group of people kneeling down beside some sheep.
Pred: a group of people standing around a dog on a leash
--------------------------------------------------------------------------------
GT:   Two skiers prepare to make their way past an embankment
Pred: a group of people riding horses through a snow covered field
--------------------------------------------------------------------------------
GT:   A person on skis skiing down a mountain.
Pred: a person skiing down a snow covered slope
--------------------------------------------------------------------------------
✅ Inference test complete
